In [ ]:
# Imports
import polars as pl
from dotenv import load_dotenv
load_dotenv()
import os
from icecream import ic
from demoparser2 import DemoParser
import matplotlib.pyplot as plt
import numpy as np
TICK_RATE = 64

In [ ]:
# Parse match
parser = DemoParser("data/raw/match.dem")
death_df = pl.from_pandas(parser.parse_event("player_death"))
ticks = pl.from_pandas(parser.parse_ticks(["X", "Y", "Z","pitch","yaw","player_steamid","tick","is_alive","round_start_time","is_freeze_period","death_time"]))
ticks = (
    ticks
    .sort(["player_steamid", "tick"])
    .with_columns(
        d_yaw_raw=pl.col("yaw").diff().over("player_steamid"),
        d_tick=pl.col("tick").diff().over("player_steamid"),
        prev_alive=pl.col("is_alive").shift(1).over("player_steamid"),
        prev_freeze=pl.col("is_freeze_period").shift(1).over("player_steamid")
    )
    .with_columns(
        d_yaw=((pl.col("d_yaw_raw") + 180) % 360) - 180,
    )
    .with_columns(
        yaw_speed=pl.when(
            (pl.col("tick") > 0) & pl.col("is_alive") & pl.col("prev_alive") & (pl.col("prev_freeze") != True)
        )
        .then(pl.col("d_yaw").abs() / pl.col("d_tick") * TICK_RATE)
        .otherwise(None),
    )
).fill_null(0)


In [ ]:
# Select player
player_info = pl.from_pandas(parser.parse_player_info())
print(*player_info["name"].to_list())
SELECTED_NAME = "a player name from your demo"
SELECTED_ID = player_info.filter(name=SELECTED_NAME)["steamid"]

In [ ]:
# Select player
player_info = pl.from_pandas(parser.parse_player_info())
print(*player_info["name"].to_list())
fig, ax = plt.subplots(figsize=(11,5))
for player in player_info["name"].to_list():
    SELECTED_ID = player_info.filter(name=player)["steamid"]
    # Get angular velocity before death
    SECONDS_BEFORE_DEATH = 2
    SECONDS_AFTER_DEATH = 0.5
    ticks_before_death = SECONDS_BEFORE_DEATH*TICK_RATE
    ticks_after_death = int(SECONDS_AFTER_DEATH*TICK_RATE)
    kill_ticks=death_df.filter(attacker_steamid=str(SELECTED_ID.to_list()[0]))["tick"].to_list()
    alive_ticks=ticks.filter(player_steamid=SELECTED_ID).drop_nulls()
    kill_windows = {}
    for i, kill_tick in enumerate(kill_ticks):
        yaw_speed = []
        for tick in range(kill_tick-ticks_before_death,kill_tick+ticks_after_death+1):
            yaw_speed.append(alive_ticks.filter(tick=tick)["yaw_speed"].to_list()[0])
        kill_windows[i]=yaw_speed
    for i in kill_windows.keys():
        ax.plot([tick/64 for tick in range(-ticks_before_death,len(kill_windows[i])-ticks_before_death)], kill_windows[i],color="tab:blue",alpha=0.3,linewidth=2)
plt.savefig("kill windows.png")
plt.show() 
# plt.hist(alive_ticks,bins=200,log=True)
# plt.show()

In [ ]:
# Select player
player_info = pl.from_pandas(parser.parse_player_info())
print(*player_info["name"].to_list())
fig, ax = plt.subplots(figsize=(11,5))
SELECTED_PLAYER = "a player name from your demo"
SELECTED_ID = player_info.filter(name=player)["steamid"]
# Get angular velocity before death
SECONDS_BEFORE_DEATH = 2
SECONDS_AFTER_DEATH = 0.5
ticks_before_death = SECONDS_BEFORE_DEATH*TICK_RATE
ticks_after_death = int(SECONDS_AFTER_DEATH*TICK_RATE)
kill_ticks=death_df.filter(attacker_steamid=str(SELECTED_ID.to_list()[0]))["tick"].to_list()
alive_ticks=ticks.filter(player_steamid=SELECTED_ID).drop_nulls()
kill_windows = {}
for i, kill_tick in enumerate(kill_ticks):
    yaw_speed = []
    for tick in range(kill_tick-ticks_before_death,kill_tick+ticks_after_death+1):
        yaw_speed.append(alive_ticks.filter(tick=tick)["yaw_speed"].to_list()[0])
    kill_windows[i]=yaw_speed
for i in kill_windows.keys():
    ax.plot([tick/64 for tick in range(-ticks_before_death,len(kill_windows[i])-ticks_before_death)], kill_windows[i],color="tab:blue",alpha=0.3,linewidth=2)
plt.savefig("kill windows.png")
plt.show() 
# plt.hist(alive_ticks,bins=200,log=True)
# plt.show()